# 04 — Q1: previsão do preço do mês seguinte

**Questão:** é possível prever o preço do mês seguinte de cada produto melhor do que simplesmente repetir o preço atual?

**Método (decisões 3, 6 e 7):**
- série: preço geral (mediana do PPK entre mercados) por produto e mês;
- entradas: preços dos meses anteriores (lags); o custo do produtor (CONAB) foi testado no notebook 03 e ficou fora dos modelos finais (decisão 5);
- modelos: baseline ingênuo, regressão linear, KNN e ensemble; o modelo extra (provavelmente Random Forest) só é decidido depois destes resultados;
- avaliação **sempre com divisão temporal** (treino nos meses antigos, teste nos recentes, ou walk-forward). **Proibido k-fold embaralhado.**
- hiperparâmetros (número de lags, k do KNN) escolhidos numa validação separada dentro do treino, nunca no teste;
- métricas em valor (MAE, MSE) e de direção sobe/não sobe (matriz de confusão, acurácia, precisão, revocação), reportadas separadamente.

Conclusão negativa (nenhum modelo bate o baseline) também é resultado e deve ser reportada com justificativa.

In [ ]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir da pasta notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
import src.features
import src.evaluation
import src.q1_models

## ⚠️ Rascunho do Passo 5 — dados fictícios

A pessoa A ainda não entregou `aggregation.serie_geral`. Pra não ficar travado,
a célula abaixo **inventa** uma série de preços (passeio aleatório) no mesmo
formato longo da decisão 9 (`produto`, `mes`, `preco`, `n_mercados`), só pra
validar que o pipeline inteiro roda de ponta a ponta.

**Quando a base real chegar, a única célula que precisa mudar é a próxima
(a que gera `geral`)** — o resto do notebook já lê `geral` sem saber se é
fictício ou real.

Ainda sem CONAB e sem ensemble aqui (Passo 5 é só baseline + regressão linear
+ KNN, pra decidir o que vem depois — Passos 6 e 7).

In [ ]:
# FICTÍCIO — substituir por `src.aggregation.serie_geral(base_limpa)` quando a
# parte A entregar.
rng = np.random.default_rng(config.SEMENTE)
meses = pd.period_range(config.PERIODO_INICIO, periods=config.TOTAL_MESES, freq="M")

linhas = []
for produto in config.PRODUTOS:
    preco_inicial = rng.uniform(2, 40)
    tendencia = rng.normal(0.05, 0.05, size=config.TOTAL_MESES)
    ruido = rng.normal(0, preco_inicial * 0.02, size=config.TOTAL_MESES)
    precos = preco_inicial + np.cumsum(tendencia + ruido)

    for mes, preco in zip(meses, precos):
        linhas.append({
            "produto": produto,
            "mes": mes,
            "preco": preco,
            "n_mercados": rng.integers(20, 35),
        })

geral = pd.DataFrame(linhas)
geral.head()

### Separar os meses de treino antes de escolher hiperparâmetros

`escolher_hiperparametros` só pode ver os meses de treino (decisão 7: nunca
olhar o teste). `avaliar_modelo`, mais abaixo, recebe a base com **todos** os
meses — é o próprio `walk_forward` dentro dela que isola os meses de teste
(37 a 48) a partir de `config.MESES_TREINO`.

In [ ]:
meses_ordenados = sorted(geral["mes"].unique())
meses_treino = meses_ordenados[:config.MESES_TREINO]
geral_treino = geral[geral["mes"].isin(meses_treino)]

len(meses_ordenados), len(meses_treino)

In [ ]:
linhas_resultado = []

for produto in config.PRODUTOS:
    hiperparametros = src.q1_models.escolher_hiperparametros(geral_treino, produto)
    n_lags = hiperparametros["n_lags"]
    k = hiperparametros["k"]

    base = src.features.montar_base_q1(geral, produto, n_lags)
    colunas_entrada = [f"lag_{i}" for i in range(1, n_lags + 1)]

    modelos = {
        "baseline": src.q1_models.BaselineIngenuo(),
        "regressao_linear": src.q1_models.criar_regressao_linear(),
        "knn": src.q1_models.criar_knn(k),
    }

    for nome_modelo, modelo in modelos.items():
        metricas = src.evaluation.avaliar_modelo(modelo, base, colunas_entrada, "alvo")
        linha = metricas.iloc[0].to_dict()
        linha.update({"produto": produto, "modelo": nome_modelo, "n_lags": n_lags, "k": k})
        linhas_resultado.append(linha)

tabela_resultados = pd.DataFrame(linhas_resultado)[
    ["produto", "modelo", "n_lags", "k", "MAE", "MSE", "acuracia", "precisao", "revocacao", "matriz_confusao"]
]
tabela_resultados.sort_values(["produto", "modelo"]).reset_index(drop=True)